## Import Libraries and CSV Data

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
#from sklearn.model_selection import GridSearchCV

df = pd.read_csv("csv's/mlb_2024-2026.csv")

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{tuple(df.columns)}")
print(f"\nDate range: {df['game_date'].min()} → {df['game_date'].max()}")
print(f"\nPitch type distribution:\n{df['pitch_type'].value_counts()}")

Shape: (1784664, 117)

Columns:
('pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'babip_value', 'iso_value', 'l

# Clean Data

In [7]:
STAT_MAP = {
    'release_spin_rate' : 'spin_rate',
    'pfx_z'             : 'induced_vertical_break',
    'pfx_x'             : 'horizontal_break',
    'release_extension' : 'extension',
    'release_pos_z'     : 'release_height',
    'release_pos_x'     : 'release_side',
    'release_spin_axis' : 'spin_axis',
}

FEATURES = [
    'release_speed',
    'spin_rate',
    'induced_vertical_break',
    'horizontal_break',
    'extension',
    'release_height',
    'release_side',
    'spin_axis',
]

FLIP_COLS = ['horizontal_break', 'release_side']

# Unnecessary pitch types
DROP_TYPES = {'SV', 'EP', 'FA', 'CS', 'SC', 'PO', 'UN'}

# Map Statcast pitch types to Trackman pitch types
PITCH_MAP = {
    'FF': 'FB',
    'CU': 'CB',
    'KC': 'CB',  # knuckle-curve treated as CB
    'FC': 'CT',
    'ST': 'SW',
    'FS': 'SP',
    'FO': 'SP',  # forkball treated as SP
}

df = df.rename(columns=STAT_MAP)

# Normalize LHP → RHP convention
df.loc[df['p_throws'] == 'L', FLIP_COLS] *= -1
df.loc[df['p_throws'] == 'L', 'spin_axis'] = (360 - df.loc[df['p_throws'] == 'L', 'spin_axis']) % 360

# Drop rows missing the target variable
df = df.dropna(subset=['delta_run_exp']).copy()

# Drop unnecessary pitch types
df = df[~df['pitch_type'].isin(DROP_TYPES)]

# Merge certain pitches
df['pitch_type'] = df['pitch_type'].replace(PITCH_MAP)

print(f"Rows after cleaning: {len(df):,}")
print(f"\nNull counts:\n{df[FEATURES + ['delta_run_exp']].isnull().sum()}")
print(f"\nPitch type counts after cleaning:\n{df['pitch_type'].value_counts()}")

Rows after cleaning: 1,765,985

Null counts:
release_speed             30452
spin_rate                 36755
induced_vertical_break    30451
horizontal_break          30584
extension                 32311
release_height            30451
release_side              30451
spin_axis                 36757
delta_run_exp                 0
dtype: int64

Pitch type counts after cleaning:
pitch_type
FB    553840
SI    276048
SL    254143
CH    181021
CB    143232
CT    137654
SW    130248
SP     57673
KN      1378
Name: count, dtype: int64


# Train Model

In [ ]:
for pitch_type, group in df.groupby('pitch_type'):
    # Define inputs and outputs
    X = group[FEATURES]
    y = group['delta_run_exp']

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 24)

    # Set hyperparameters (Use gridsearch later to tune these better)
    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=24
        )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    # Uncomment this line to save the models again if necessary
    #model.save_model(f'models/{pitch_type}.ubj')

    print(f"{pitch_type}: n = {len(group):,} RMSE = {rmse:.4f}")

print("\nModels saved to /models directory")

CB: n = 143,232 RMSE = 0.2111
CH: n = 181,021 RMSE = 0.2233
CT: n = 137,654 RMSE = 0.2398
FB: n = 553,840 RMSE = 0.2278
KN: n = 1,378 RMSE = 0.2090
SI: n = 276,048 RMSE = 0.2257
SL: n = 254,143 RMSE = 0.2247
SP: n = 57,673 RMSE = 0.2187
SW: n = 130,248 RMSE = 0.2212

Models saved to /models directory
